In [1]:
# =========================
# CELL 0) Load graph G from DB
# =========================
import sys
from pathlib import Path

REPO_ROOT = Path(r"C:\Users\Ahmed sameh\Downloads\mapedia-master")
sys.path.insert(0, str(REPO_ROOT))
sys.modules.pop("modules", None)

from modules import DBHandler

db_handler = DBHandler()
db_handler.connect_to_db()

G = db_handler.get_graph(
    min_lat= 40.4774,
    max_lat= 40.9176,
    min_lon= -74.2591,
    max_lon= -73.7004
)

print(type(G), len(G.nodes), len(G.edges))


<class 'networkx.classes.multidigraph.MultiDiGraph'> 809662 1146915


In [2]:
# =========================
# CELL 1) DATA (DB graph -> gdf_edges) + merge road_attributes  (+ keep length)
# =========================
import numpy as np
import pandas as pd
import geopandas as gpd
import ast

from sqlalchemy import create_engine

import osmnx as ox
gdf_nodes, gdf_edges = ox.graph_to_gdfs(G)
gdf_edges = gdf_edges.reset_index()  # u,v,key as columns

print("Graph type:", type(G))
print("Nodes:", len(G.nodes), "Edges:", len(G.edges))
print("gdf_edges columns:", list(gdf_edges.columns))
print("gdf_edges CRS:", gdf_edges.crs)

# --- DB connection ---
DB_HOST = 'cs-u-spatial-406.cs.umn.edu'
DB_NAME = 'gis'
DB_USER = 'gis'
DB_PASS = 'gis'
DB_PORT = 5432
engine = create_engine(f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

# 1) Fetch road_attributes for osmids present in current gdf_edges
osmids = gdf_edges["osmid"].astype(np.int64).unique().tolist()
print("Unique osmids in graph:", len(osmids))

def fetch_road_attributes_in_chunks(id_list, chunk_size=200000):
    out = []
    for i in range(0, len(id_list), chunk_size):
        chunk = id_list[i:i+chunk_size]
        ids_tuple = str(tuple(chunk)).replace(",)", ")")
        q = f"""
            SELECT osm_id, road_type, width, nlanes, max_speed, min_speed
            FROM road_attributes
            WHERE osm_id IN {ids_tuple}
        """
        out.append(pd.read_sql(q, engine))
        print(f"Fetched road_attributes chunk {i}..{min(i+chunk_size, len(id_list))}")
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame()

ra = fetch_road_attributes_in_chunks(osmids)
print("road_attributes fetched rows:", len(ra))
print("road_attributes unique osm_id:", ra["osm_id"].nunique())

# 2) Merge (left) onto edges by osmid
ra = ra.rename(columns={"osm_id": "osmid"})
merged = gdf_edges.merge(ra, on="osmid", how="left", validate="many_to_one")

print("After merge, null counts:")
print(merged[["road_type","width","nlanes"]].isnull().sum())

# 3) Cleaning utilities for DB numeric fields (width, nlanes)
def _parse_item_as_float(item):
    s = str(item).lower().replace("m", "").strip()
    if s in {"nan", "none", ""}:
        return np.nan
    if ";" in s:
        s = s.split(";")[0].strip()
    try:
        return float(s)
    except Exception:
        print("Error") # Don't silently break
        return np.nan

def clean_width_db(v):
    if pd.isna(v):
        return np.nan
    if isinstance(v, (list, tuple)):
        vals = [_parse_item_as_float(x) for x in v]
        vals = [x for x in vals if not np.isnan(x)]
        return float(np.mean(vals)) if vals else np.nan
    s = str(v).strip()
    if s.startswith("["):
        try:
            lst = ast.literal_eval(s)
            vals = [_parse_item_as_float(x) for x in lst]
            vals = [x for x in vals if not np.isnan(x)]
            return float(np.mean(vals)) if vals else np.nan
        except Exception:
            print("Error in Width")
            return np.nan
    return _parse_item_as_float(v)

def clean_nlanes_db(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s.startswith("["):
        try:
            lst = ast.literal_eval(s)
            return int(_parse_item_as_float(lst[0]))
        except Exception:
            print("Error in Lanes")
            return np.nan
    x = _parse_item_as_float(v)
    return int(x) if not np.isnan(x) else np.nan

# 4) Clean oneway from gdf_edges (strings like YES/NO) -> 0/1
def clean_oneway_edge(v):
    if pd.isna(v):
        return pd.NA

    if isinstance(v, (bool, np.bool_)):
        return int(v)

    s = str(v).strip().lower()

    # forward one-way
    if s in {"yes", "true", "1"}:
        return 1

    # reverse-direction one-way (still one-way, just opposite orientation)
    if s in {"-1", "reverse", "reversed"}:
        return 1

    # two-way
    if s in {"no", "false", "0"}:
        return 0

    # unknown / dynamic / empty
    if s in {"unknown", "nan", "none", "", "reversible", "alternating"}:
        return pd.NA

    # fallback numeric parse
    try:
        x = float(s)
        if x == -1.0:
            return 1
        if x == 0.0:
            return 0
        if x == 1.0:
            return 1
        return pd.NA
    except Exception:
        print("Error in One Way")
        return pd.NA


final_gdf = merged.copy()
final_gdf["oneway"] = final_gdf["oneway"].apply(clean_oneway_edge).astype("Int64")
final_gdf["width"]  = final_gdf["width"].apply(clean_width_db)
final_gdf["lanes"]  = final_gdf["nlanes"].apply(clean_nlanes_db)  # rename to lanes
final_gdf["length"] = pd.to_numeric(final_gdf["length"], errors="coerce")
final_gdf["max_speed"] = pd.to_numeric(final_gdf["max_speed"], errors="coerce")
final_gdf["min_speed"] = pd.to_numeric(final_gdf["min_speed"], errors="coerce")
# 5) Keep only needed columns (+ length + geometry and u/v for line graph)
keep_cols = ["u","v","key","id","osmid","oneway","road_type","lanes","width","length","geometry","max_speed","min_speed"]
missing = [c for c in keep_cols if c not in final_gdf.columns]
if missing:
    raise ValueError(f"Missing expected columns after merge/clean: {missing}")

final_gdf = gpd.GeoDataFrame(final_gdf[keep_cols], geometry="geometry", crs=gdf_edges.crs)
final_gdf = final_gdf.rename(columns={"road_type": "highway"})

print("\nfinal_gdf info:")
print(final_gdf.info())

print("\nMissing values:")
print(final_gdf[["highway","lanes","width","oneway","length","max_speed","min_speed"]].isnull().sum())


Graph type: <class 'networkx.classes.multidigraph.MultiDiGraph'>
Nodes: 809662 Edges: 1146915
gdf_edges columns: ['u', 'v', 'key', 'id', 'osmid', 'maxspeed', 'oneway', 'length', 'geometry']
gdf_edges CRS: EPSG:4326
Unique osmids in graph: 489513
Fetched road_attributes chunk 0..200000
Fetched road_attributes chunk 200000..400000
Fetched road_attributes chunk 400000..489513
road_attributes fetched rows: 213300
road_attributes unique osm_id: 213300
After merge, null counts:
road_type     625623
width        1144009
nlanes       1035066
dtype: int64

final_gdf info:
<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 1146915 entries, 0 to 1146914
Data columns (total 13 columns):
 #   Column     Non-Null Count    Dtype   
---  ------     --------------    -----   
 0   u          1146915 non-null  int64   
 1   v          1146915 non-null  int64   
 2   key        1146915 non-null  int64   
 3   id         1146915 non-null  int64   
 4   osmid      1146915 non-null  int64   
 5   one

In [3]:
# =========================
# 0) Imports
# =========================
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import geopandas as gpd

from torch_geometric.data import Data
from torch_geometric.nn import GATv2Conv

# Reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================
# 1) Helpers: parsing & encoding
# =========================
def lanes_to_class(x):
    """
    x: numeric lanes value or NaN
    returns:
      -1 if NaN
       0 for 1 lane
       1 for 2 lanes
       2 for 3+ lanes
    """
    if pd.isna(x):
        return -1
    v = float(x)
    if v <= 1.0:
        return 0
    elif v <= 2.0:
        return 1
    else:
        return 2

class ZScaler:
    """Z-score scaler that ignores NaN."""
    def __init__(self):
        self.mu = None
        self.sd = None

    def fit(self, x: np.ndarray):
        self.mu = np.nanmean(x)
        self.sd = np.nanstd(x) + 1e-8

    def transform(self, x: np.ndarray):
        return (x - self.mu) / self.sd

# =========================
# 2) Build line graph adjacency from (u,v)
# =========================
def build_line_graph_edge_index(df, u_col="u", v_col="v", eid_col="eid"):
    """
    Nodes in line graph = edges in original graph (df rows).
    Two line-graph nodes connect if original edges share an endpoint (u or v).
    """
    incident = {}
    u_vals = df[u_col].values
    v_vals = df[v_col].values
    eids = df[eid_col].values

    for u, v, eid in zip(u_vals, v_vals, eids):
        incident.setdefault(u, []).append(eid)
        incident.setdefault(v, []).append(eid)

    pairs = []
    for _, lst in incident.items():
        m = len(lst)
        if m < 2:
            continue
        for i in range(m):
            for j in range(i + 1, m):
                a, b = lst[i], lst[j]
                pairs.append((a, b))
                pairs.append((b, a))

    if len(pairs) == 0:
        return torch.empty((2, 0), dtype=torch.long)

    edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()
    edge_index = torch.unique(edge_index, dim=1)  # remove duplicate directed edges
    return edge_index


# =========================
# 3) Prepare dataframe: ids, parsing, vocab
# =========================
df = final_gdf.copy().reset_index(drop=True)
df["eid"] = np.arange(len(df), dtype=np.int64)

# --- ensure we have a geometry-derived centroid (for spatial split) ---
if "geometry" not in df.columns:
    raise ValueError("final_gdf is missing 'geometry'. Needed for spatial split. "
                     "Ensure ox.graph_to_gdfs(G) produced edge geometries.")

df = gpd.GeoDataFrame(df, geometry="geometry")
if df.crs is None:
    df = df.set_crs(epsg=4326)

df = df[~df["geometry"].isna()].copy()
# targets/features
df["width_m"] = df["width"]
df["lanes_cls"] = df["lanes"].apply(lanes_to_class).astype(np.int64)
df["max_speed_val"] = df["max_speed"]
df["min_speed_val"] = df["min_speed"]

oneway_series = df["oneway"]
df["oneway01"] = oneway_series.astype("Float64")  # 0/1/NA

MASK_TOKEN = "__MASK__"
UNK_TOKEN = "__UNK__"

# If your column name is road_type instead of highway, swap here.
# This cell assumes "highway" exists like your original; if you're using road_type, change df["highway"] to df["road_type"].
if "highway" not in df.columns:
    raise ValueError("Expected a 'highway' column (like your original cell). "
                     "If you're using road_type, replace 'highway' with 'road_type' in this cell.")

highway_vals = df["highway"].fillna(UNK_TOKEN).astype(str).values
unique_highways = sorted(pd.unique(highway_vals).tolist())

if UNK_TOKEN not in unique_highways:
    unique_highways.append(UNK_TOKEN)
if MASK_TOKEN in unique_highways:
    unique_highways.remove(MASK_TOKEN)
unique_highways.append(MASK_TOKEN)

hwy2id = {h: i for i, h in enumerate(unique_highways)}
id2hwy = {i: h for h, i in hwy2id.items()}
df["highway_id"] = pd.Series(highway_vals).map(lambda x: hwy2id.get(x, hwy2id[UNK_TOKEN])).astype(np.int64)

HIGHWAY_MASK_ID = hwy2id[MASK_TOKEN]

LANES_MASK_ID = 3   # lanes classes: 0,1,2 + MASK=3 + MISSING=4
LANES_MISS_ID = 4

# IMPORTANT CHANGE vs your original:
# oneway embedding now supports a MISSING token too:
# 0,1 + MASK=2 + MISSING=3
ONEWAY_MASK_ID = 2
ONEWAY_MISS_ID = 3

# =========================
# 4) SPATIAL split — longitude-quantile bands (simple, contiguous, no leakage)
# =========================
N = len(df)
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.80, 0.10, 0.10
SPLIT_AXIS = "lon"   # "lon" = cut along longitude (x), "lat" = along latitude (y)

centroids = df.geometry.centroid
coord = (centroids.x if SPLIT_AXIS == "lon" else centroids.y).to_numpy()
order = np.argsort(coord, kind="stable")

n_train = int(round(TRAIN_FRAC * N))
n_val   = int(round(VAL_FRAC   * N))

train_idx = np.sort(order[:n_train]).astype(np.int64)
val_idx   = np.sort(order[n_train : n_train + n_val]).astype(np.int64)
test_idx  = np.sort(order[n_train + n_val :]).astype(np.int64)

print(f"[Spatial split] axis={SPLIT_AXIS}  train={len(train_idx)}  val={len(val_idx)}  test={len(test_idx)}")

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0

# =========================
# 5) Normalize continuous features (fit train-only)
# =========================
# KEEP length as an INPUT feature, but DO NOT create y_length and DO NOT predict it.

length_raw = df["length"].to_numpy(dtype=np.float32)
length_log = np.log1p(length_raw)

width_raw = df["width_m"].to_numpy(dtype=np.float32)  # may contain NaN
max_raw = df["max_speed_val"].to_numpy(dtype=np.float32)
min_raw = df["min_speed_val"].to_numpy(dtype=np.float32)

len_scaler = ZScaler()
wid_scaler = ZScaler()
max_scaler = ZScaler()
min_scaler = ZScaler()

len_scaler.fit(length_log[train_idx])
wid_scaler.fit(width_raw[train_idx])
max_scaler.fit(max_raw[train_idx])
min_scaler.fit(min_raw[train_idx])

length_z = len_scaler.transform(length_log).astype(np.float32)
width_z  = wid_scaler.transform(width_raw).astype(np.float32)  # NaN stays NaN
max_z = max_scaler.transform(max_raw).astype(np.float32)
min_z = min_scaler.transform(min_raw).astype(np.float32)

width_missing  = np.isnan(width_z).astype(np.float32)
length_missing = np.zeros_like(width_missing, dtype=np.float32)  # length is complete usually
max_missing = np.isnan(max_z).astype(np.float32)
min_missing = np.isnan(min_z).astype(np.float32)

width_z = np.nan_to_num(width_z, nan=0.0)
max_z = np.nan_to_num(max_z, nan=0.0)
min_z = np.nan_to_num(min_z, nan=0.0)

# continuous features for ALL nodes:
# [length_z, width_z, length_missing, width_missing, len_masked_flag, wid_masked_flag]
x_cont_all = np.column_stack([
    length_z,
    width_z,
    max_z,
    min_z,
    length_missing,
    width_missing,
    max_missing,
    min_missing,
    np.zeros(N, dtype=np.float32),  # len_mask
    np.zeros(N, dtype=np.float32),  # wid_mask
    np.zeros(N, dtype=np.float32),  # max_mask
    np.zeros(N, dtype=np.float32),  # min_mask
]).astype(np.float32)

# =========================
# 6) Build induced subgraph per split (spatial-inductive)
# =========================
edge_index_full = build_line_graph_edge_index(df, u_col="u", v_col="v", eid_col="eid")

ei = edge_index_full
pairs = ei[0].cpu().numpy().astype(np.int64) * (ei.max().item() + 1) + ei[1].cpu().numpy().astype(np.int64)
dup = len(pairs) - len(np.unique(pairs))
print("duplicate directed edges in line-graph edge_index:", dup)

edge_index_full_np = edge_index_full.cpu().numpy()

# Global targets (NO LENGTH TARGET)
y_highway_all = df["highway_id"].to_numpy(dtype=np.int64)
y_lanes_all   = df["lanes_cls"].to_numpy(dtype=np.int64)        # -1 for missing
y_oneway_all  = df["oneway01"].to_numpy(dtype=np.float32)       # contains NaN for unknown
y_width_all   = df["width_m"].to_numpy(dtype=np.float32)        # may be NaN
y_max_all = df["max_speed_val"].to_numpy(dtype=np.float32)
y_min_all = df["min_speed_val"].to_numpy(dtype=np.float32)
lanes_in_all = df["lanes_cls"].to_numpy(dtype=np.int64)
lanes_in_all = np.where(lanes_in_all == -1, LANES_MISS_ID, lanes_in_all).astype(np.int64)

# oneway input ids with MISSING token
oneway_in_all = df["oneway01"].to_numpy(dtype=np.float32)  # 0/1/NaN
oneway_in_all = np.where(np.isnan(oneway_in_all), ONEWAY_MISS_ID, oneway_in_all).astype(np.int64)

def build_split_data(split_idx_np):
    split_idx_np = np.array(split_idx_np, dtype=np.int64)
    split_idx_np = np.unique(split_idx_np)
    split_idx_np.sort()

    map_arr = np.full(N, -1, dtype=np.int64)
    map_arr[split_idx_np] = np.arange(len(split_idx_np), dtype=np.int64)

    src_old = edge_index_full_np[0]
    dst_old = edge_index_full_np[1]

    keep = (map_arr[src_old] >= 0) & (map_arr[dst_old] >= 0)
    src_new = map_arr[src_old[keep]]
    dst_new = map_arr[dst_old[keep]]

    edge_index = torch.from_numpy(np.stack([src_new, dst_new], axis=0)).long()

    data = Data(
        x_cont=torch.from_numpy(x_cont_all[split_idx_np]).float(),
        edge_index=edge_index
    )
    data.num_nodes = len(split_idx_np)
    data.x = data.x_cont

    # inputs
    data.highway_in = torch.from_numpy(y_highway_all[split_idx_np]).long()
    data.lanes_in   = torch.from_numpy(lanes_in_all[split_idx_np]).long()
    data.oneway_in  = torch.from_numpy(oneway_in_all[split_idx_np]).long()

    # targets (NO y_length)
    data.y_highway = torch.from_numpy(y_highway_all[split_idx_np]).long()
    data.y_lanes   = torch.from_numpy(y_lanes_all[split_idx_np]).long()
    data.y_oneway  = torch.from_numpy(y_oneway_all[split_idx_np]).float()
    data.y_width   = torch.from_numpy(y_width_all[split_idx_np]).float()
    data.y_max = torch.from_numpy(y_max_all[split_idx_np]).float()
    data.y_min = torch.from_numpy(y_min_all[split_idx_np]).float()
    return data.to(device)

data_train = build_split_data(train_idx)
data_val   = build_split_data(val_idx)
data_test  = build_split_data(test_idx)

print("Train graph:", data_train.num_nodes, "nodes |", data_train.edge_index.shape[1], "edges")
print("Val graph:  ", data_val.num_nodes, "nodes |", data_val.edge_index.shape[1], "edges")
print("Test graph: ", data_test.num_nodes, "nodes |", data_test.edge_index.shape[1], "edges")

def degree_stats(data):
    deg = torch.bincount(data.edge_index[0], minlength=data.num_nodes)
    return {
        "num_nodes": int(data.num_nodes),
        "isolated": int((deg == 0).sum().item()),
        "deg1": int((deg == 1).sum().item()),
        "mean_deg": float(deg.float().mean().item()),
    }

print("Degree stats:")
print("  Train:", degree_stats(data_train))
print("  Val:  ", degree_stats(data_val))
print("  Test: ", degree_stats(data_test))

# =========================
# 7) Model: embeddings + GATv2 + heads + learnable loss weights
# =========================
class MultiAttrGAT(nn.Module):
    def __init__(self, num_highway, hwy_emb_dim=16,
                 lanes_emb_dim=8, oneway_emb_dim=4,
                 cont_dim=12, hidden=32, heads=2, dropout=0.1):
        super().__init__()

        self.hwy_emb = nn.Embedding(num_highway, hwy_emb_dim)
        self.lanes_emb = nn.Embedding(5, lanes_emb_dim)    # 0,1,2, MASK=3, MISS=4
        self.oneway_emb = nn.Embedding(4, oneway_emb_dim)  # 0,1, MASK=2, MISS=3 

        in_dim = cont_dim + hwy_emb_dim + lanes_emb_dim + oneway_emb_dim

        self.gat1 = GATv2Conv(in_dim, hidden, heads=heads, concat=True, dropout=dropout)
        self.gat2 = GATv2Conv(hidden * heads, hidden, heads=heads, concat=False, dropout=dropout)

        self.head_highway = nn.Linear(hidden, num_highway)
        self.head_lanes   = nn.Linear(hidden, 3)
        self.head_oneway  = nn.Linear(hidden, 1)
        self.head_width   = nn.Linear(hidden, 1)
        self.head_max = nn.Linear(hidden, 1)
        self.head_min = nn.Linear(hidden, 1)
        # total = Σ exp(-s_i)*L_i + s_i; order: [highway, lanes, oneway, width, max, min]
        self.log_vars = nn.Parameter(torch.zeros(6))

    def forward(self, x_cont, highway_in, lanes_in, oneway_in, edge_index):
        hwy = self.hwy_emb(highway_in)
        lan = self.lanes_emb(lanes_in)
        onw = self.oneway_emb(oneway_in)

        x = torch.cat([x_cont, hwy, lan, onw], dim=1)

        h = F.elu(self.gat1(x, edge_index))
        h = F.elu(self.gat2(h, edge_index))

        return {
            "highway": self.head_highway(h),
            "lanes": self.head_lanes(h),
            "oneway": self.head_oneway(h).squeeze(-1),
            "width": self.head_width(h).squeeze(-1),
            "max_speed": self.head_max(h).squeeze(-1),
            "min_speed": self.head_min(h).squeeze(-1),
        }

    def weighted_sum(self, losses_dict):
        L = torch.stack([
            losses_dict["hwy"],
            losses_dict["lan"],
            losses_dict["onw"],
            losses_dict["wid"],
            losses_dict["max"],
            losses_dict["min"],
        ])
        precision = torch.exp(-self.log_vars)
        return torch.sum(precision * L + self.log_vars)

num_highway = len(hwy2id)
model = MultiAttrGAT(num_highway=num_highway, cont_dim=12).to(device)

loss_ce    = nn.CrossEntropyLoss()
loss_bce   = nn.BCEWithLogitsLoss()
loss_huber = nn.SmoothL1Loss()

# =========================
# 8) Masking utils (per-graph)
# =========================
def bernoulli_mask(num_nodes, valid_mask, p=0.3):
    r = torch.rand(num_nodes, device=valid_mask.device)
    return (r < p) & valid_mask

CONT_LENGTH_COL   = 0
CONT_WIDTH_COL    = 1
CONT_MAX_COL      = 2
CONT_MIN_COL      = 3

CONT_LENMISS_COL  = 4
CONT_WIDMISS_COL  = 5
CONT_MAXMISS_COL  = 6
CONT_MINMISS_COL  = 7

CONT_LENMASK_COL  = 8
CONT_WIDMASK_COL  = 9
CONT_MAXMASK_COL  = 10
CONT_MINMASK_COL  = 11

def make_fixed_masks(data, p_mask, seed=999):
    gen = torch.Generator(device=data.y_highway.device)
    gen.manual_seed(seed)

    n = data.num_nodes
    valid_hwy = torch.ones(n, dtype=torch.bool, device=data.y_highway.device)
    valid_lan = (data.y_lanes != -1)

    # oneway: only valid where we know label (not NaN)
    valid_onw = ~torch.isnan(data.y_oneway)

    valid_wid = ~torch.isnan(data.y_width)
    valid_max = ~torch.isnan(data.y_max)
    valid_min = ~torch.isnan(data.y_min)
    def fixed_mask(valid_mask, p):
        r = torch.rand(n, generator=gen, device=valid_mask.device)
        return (r < p) & valid_mask

    return {
        "hwy": fixed_mask(valid_hwy, p_mask),
        "lan": fixed_mask(valid_lan, p_mask),
        "onw": fixed_mask(valid_onw, p_mask),
        "wid": fixed_mask(valid_wid, p_mask),
        "max": fixed_mask(valid_max, p_mask),
        "min": fixed_mask(valid_min, p_mask),
        # NOTE: no "len" mask (length is input-only)
    }

# =========================
# 9) Corruption + losses + metrics
# =========================
def corrupt_inputs_with_flags(data, masks):
    x_cont = data.x_cont.clone()
    highway_in = data.highway_in.clone()
    lanes_in = data.lanes_in.clone()
    oneway_in = data.oneway_in.clone()

    highway_in[masks["hwy"]] = HIGHWAY_MASK_ID
    lanes_in[masks["lan"]]   = LANES_MASK_ID
    oneway_in[masks["onw"]]  = ONEWAY_MASK_ID

    # reset flags
    x_cont[:, CONT_LENMASK_COL] = 0.0
    x_cont[:, CONT_WIDMASK_COL] = 0.0
    x_cont[:, CONT_MAXMASK_COL] = 0.0
    x_cont[:, CONT_MINMASK_COL] = 0.0

    # IMPORTANT: we still can mask LENGTH as an *input corruption* channel to regularize

    # If we want length to be masked sometimes as a denoising input, we do it here:
    # Example: mask length whenever width is masked (keeps code structure). You can comment it out.
    # x_cont[masks["wid"], CONT_LENGTH_COL] = 0.0
    # x_cont[masks["wid"], CONT_LENMASK_COL] = 1.0

    x_cont[masks["wid"], CONT_WIDTH_COL]  = 0.0
    x_cont[masks["wid"], CONT_WIDMASK_COL] = 1.0

    x_cont[masks["max"], CONT_MAX_COL] = 0.0
    x_cont[masks["max"], CONT_MAXMASK_COL] = 1.0

    x_cont[masks["min"], CONT_MIN_COL] = 0.0
    x_cont[masks["min"], CONT_MINMASK_COL] = 1.0

    return x_cont, highway_in, lanes_in, oneway_in

def compute_losses(pred, data, masks):
    losses = {}

    losses["hwy"] = loss_ce(pred["highway"][masks["hwy"]], data.y_highway[masks["hwy"]]) if masks["hwy"].any() \
        else torch.tensor(0.0, device=device)

    losses["lan"] = loss_ce(pred["lanes"][masks["lan"]], data.y_lanes[masks["lan"]]) if masks["lan"].any() \
        else torch.tensor(0.0, device=device)

    losses["onw"] = loss_bce(pred["oneway"][masks["onw"]], data.y_oneway[masks["onw"]]) if masks["onw"].any() \
        else torch.tensor(0.0, device=device)

    losses["wid"] = loss_huber(pred["width"][masks["wid"]], data.y_width[masks["wid"]]) if masks["wid"].any() \
        else torch.tensor(0.0, device=device)

    losses["max"] = loss_huber(pred["max_speed"][masks["max"]], data.y_max[masks["max"]]) if masks["max"].any() \
        else torch.tensor(0.0, device=device)
    
    losses["min"] = loss_huber(pred["min_speed"][masks["min"]], data.y_min[masks["min"]]) if masks["min"].any() \
        else torch.tensor(0.0, device=device)
    
    total = model.weighted_sum(losses)
    return total, losses

def macro_f1_from_preds(y_true, y_pred, num_classes):
    f1s = []
    for c in range(num_classes):
        tp = ((y_pred == c) & (y_true == c)).sum().float()
        fp = ((y_pred == c) & (y_true != c)).sum().float()
        fn = ((y_pred != c) & (y_true == c)).sum().float()

        denom_p = tp + fp
        denom_r = tp + fn
        prec = tp / denom_p if denom_p > 0 else torch.tensor(0.0, device=y_true.device)
        rec  = tp / denom_r if denom_r > 0 else torch.tensor(0.0, device=y_true.device)

        denom_f = prec + rec
        f1 = (2 * prec * rec / denom_f) if denom_f > 0 else torch.tensor(0.0, device=y_true.device)
        f1s.append(f1)

    return float(torch.stack(f1s).mean())

def binary_auroc(y_true, scores):
    y_true = y_true.float()
    scores = scores.float()

    n_pos = (y_true == 1).sum().item()
    n_neg = (y_true == 0).sum().item()
    if n_pos == 0 or n_neg == 0:
        return np.nan

    sorted_scores, order = torch.sort(scores)
    ranks = torch.empty_like(order, dtype=torch.float32)
    ranks[order] = torch.arange(1, len(scores) + 1, device=scores.device, dtype=torch.float32)

    diffs = torch.diff(sorted_scores)
    tie_starts = torch.where(diffs != 0)[0] + 1
    boundaries = torch.cat([
        torch.tensor([0], device=scores.device),
        tie_starts,
        torch.tensor([len(scores)], device=scores.device),
    ])

    for i in range(len(boundaries) - 1):
        a = int(boundaries[i].item())
        b = int(boundaries[i + 1].item())
        if b - a > 1:
            avg = (a + 1 + b) / 2.0
            ranks[order[a:b]] = avg

    sum_ranks_pos = ranks[y_true == 1].sum()
    n_pos_t = torch.tensor(float(n_pos), device=scores.device)
    n_neg_t = torch.tensor(float(n_neg), device=scores.device)

    auroc = (sum_ranks_pos - n_pos_t * (n_pos_t + 1) / 2.0) / (n_pos_t * n_neg_t)
    return float(auroc)

def compute_metrics(pred, data, masks, num_highway_classes):
    out = {}

    if masks["hwy"].any():
        y_true = data.y_highway[masks["hwy"]]
        y_pred = pred["highway"][masks["hwy"]].argmax(dim=1)
        out["hwy_macro_f1"] = macro_f1_from_preds(y_true, y_pred, num_highway_classes)
    else:
        out["hwy_macro_f1"] = np.nan

    if masks["lan"].any():
        y_true = data.y_lanes[masks["lan"]]
        y_pred = pred["lanes"][masks["lan"]].argmax(dim=1)
        out["lan_macro_f1"] = macro_f1_from_preds(y_true, y_pred, 3)
    else:
        out["lan_macro_f1"] = np.nan

    if masks["onw"].any():
        out["onw_auroc"] = binary_auroc(data.y_oneway[masks["onw"]], pred["oneway"][masks["onw"]])
    else:
        out["onw_auroc"] = np.nan

    out["wid_mae_m"] = float(torch.mean(torch.abs(pred["width"][masks["wid"]] - data.y_width[masks["wid"]]))) \
        if masks["wid"].any() else np.nan
    out["max_mae"] = float(torch.mean(torch.abs(pred["max_speed"][masks["max"]] - data.y_max[masks["max"]]))) \
        if masks["max"].any() else np.nan   
    out["min_mae"] = float(torch.mean(torch.abs(pred["min_speed"][masks["min"]] - data.y_min[masks["min"]]))) \
        if masks["min"].any() else np.nan
    # NOTE: no length metric (length is input-only)

    return out

@torch.no_grad()
def evaluate_with_masks(model, data, masks, num_highway_classes):
    model.eval()
    x_cont, highway_in, lanes_in, oneway_in = corrupt_inputs_with_flags(data, masks)
    pred = model(x_cont, highway_in, lanes_in, oneway_in, data.edge_index)
    total, losses = compute_losses(pred, data, masks)
    metrics = compute_metrics(pred, data, masks, num_highway_classes)
    return total.item(), {k: v.item() for k, v in losses.items()}, metrics

@torch.no_grad()
def evaluate_losses_only(model, data, masks):
    model.eval()
    x_cont, highway_in, lanes_in, oneway_in = corrupt_inputs_with_flags(data, masks)
    pred = model(x_cont, highway_in, lanes_in, oneway_in, data.edge_index)
    total, losses = compute_losses(pred, data, masks)
    return total.item(), {k: v.item() for k, v in losses.items()}

# =========================
# 10) Training loop + logging
# =========================
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 10000
P_MASK = 0.30
EVAL_EVERY = 50

val_masks_fixed = make_fixed_masks(data_val, p_mask=P_MASK, seed=999)

history = {
    "epoch": [],
    "train_total": [],
    "val_total": [],
    "train_losses": {k: [] for k in ["hwy", "lan", "onw", "wid", "max", "min"]},
    "val_losses":   {k: [] for k in ["hwy", "lan", "onw", "wid", "max", "min"]},

    "metric_epoch": [],
    "train_metrics": {k: [] for k in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae"]},
    "val_metrics":   {k: [] for k in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae"]},

    "log_vars": [],
}

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()

    n = data_train.num_nodes

    valid_hwy = torch.ones(n, dtype=torch.bool, device=device)
    valid_lan = (data_train.y_lanes != -1)
    valid_onw = ~torch.isnan(data_train.y_oneway)
    valid_wid = ~torch.isnan(data_train.y_width)
    valid_max = ~torch.isnan(data_train.y_max)
    valid_min = ~torch.isnan(data_train.y_min)

    train_masks = {
        "hwy": bernoulli_mask(n, valid_hwy, P_MASK),
        "lan": bernoulli_mask(n, valid_lan, P_MASK),
        "onw": bernoulli_mask(n, valid_onw, P_MASK),
        "wid": bernoulli_mask(n, valid_wid, P_MASK),
        "max": bernoulli_mask(n, valid_max, P_MASK),
        "min": bernoulli_mask(n, valid_min, P_MASK),
    }

    x_cont, highway_in, lanes_in, oneway_in = corrupt_inputs_with_flags(data_train, train_masks)
    pred = model(x_cont, highway_in, lanes_in, oneway_in, data_train.edge_index)

    total_loss, losses = compute_losses(pred, data_train, train_masks)
    total_loss.backward()
    optimizer.step()

    val_total, val_losses = evaluate_losses_only(model, data_val, val_masks_fixed)

    history["epoch"].append(epoch)
    history["train_total"].append(total_loss.item())
    history["val_total"].append(val_total)

    for k in ["hwy", "lan", "onw", "wid", "max", "min"]:
        history["train_losses"][k].append(losses[k].item())
        history["val_losses"][k].append(val_losses[k])

    history["log_vars"].append(model.log_vars.detach().cpu().numpy().copy())

    do_metrics = (epoch == 1) or (epoch % EVAL_EVERY == 0)
    if do_metrics:
        train_metrics = compute_metrics(pred, data_train, train_masks, num_highway)
        _, _, val_metrics = evaluate_with_masks(model, data_val, val_masks_fixed, num_highway)

        history["metric_epoch"].append(epoch)
        for k in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae"]:
            history["train_metrics"][k].append(train_metrics[k])
            history["val_metrics"][k].append(val_metrics[k])

        print(
            f"Epoch {epoch:04d} | total={total_loss.item():.4f} | "
            f"hwy={losses['hwy'].item():.3f}, lan={losses['lan'].item():.3f}, onw={losses['onw'].item():.3f}, "
            f"wid={losses['wid'].item():.3f} max={losses['max'].item():.3f} min={losses['min'].item():.3f} | "
            f"VAL fixed: hwy_F1={val_metrics['hwy_macro_f1']:.3f}, lan_F1={val_metrics['lan_macro_f1']:.3f}, "
            f"onw_AUROC={val_metrics['onw_auroc']:.3f}, wid_MAE(m)={val_metrics['wid_mae_m']:.3f} "
            f"max_MAE={val_metrics['max_mae']:.3f}, min_MAE={val_metrics['min_mae']:.3f} | "
            f"log_vars={model.log_vars.detach().cpu().numpy()}"
        )

# =========================
# 11) Plots
# =========================
plt.figure()
plt.plot(history["epoch"], history["train_total"], label="Train")
plt.plot(history["epoch"], history["val_total"], label="Val")
plt.xlabel("Epoch")
plt.ylabel("Total Loss")
plt.title("Total Loss (Train vs Val)")
plt.legend()
plt.show()

task_titles = {
    "hwy": "Highway CE",
    "lan": "Lanes CE",
    "onw": "Oneway BCE",
    "wid": "Width Huber",
    "max": "Max Speed Huber",
    "min": "Min Speed Huber",
}
for t in ["hwy", "lan", "onw", "wid", "max", "min"]:
    plt.figure()
    plt.plot(history["epoch"], history["train_losses"][t], label="Train")
    plt.plot(history["epoch"], history["val_losses"][t], label="Val")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{task_titles[t]} (masked-only)")
    plt.legend()
    plt.show()

metric_titles = {
    "hwy_macro_f1": "Highway Macro-F1 (masked)",
    "lan_macro_f1": "Lanes Macro-F1 (masked)",
    "onw_auroc": "Oneway AUROC (masked)",
    "wid_mae_m": "Width MAE (m, masked)",
    "max_mae": "Max Speed MAE (masked)",
    "min_mae": "Min Speed MAE (masked)",
}
for m in ["hwy_macro_f1", "lan_macro_f1", "onw_auroc", "wid_mae_m", "max_mae", "min_mae"]:
    plt.figure()
    plt.plot(history["metric_epoch"], history["train_metrics"][m], label="Train")
    plt.plot(history["metric_epoch"], history["val_metrics"][m], label="Val")
    plt.xlabel("Epoch")
    plt.ylabel(m)
    plt.title(metric_titles[m])
    plt.legend()
    plt.show()

# =========================
# 12) Final test eval (fixed test masks) on test induced graph
# =========================
test_masks_fixed = make_fixed_masks(data_test, p_mask=P_MASK, seed=2025)
test_total, test_losses, test_metrics = evaluate_with_masks(model, data_test, test_masks_fixed, num_highway)
print("TEST fixed-mask metrics:", test_metrics)
print("TEST fixed-mask losses:", test_losses)

c:\Users\Ahmed sameh\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


C:\Users\Ahmed sameh\AppData\Local\Temp\ipykernel_1464\4183623045.py:160: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = df.geometry.centroid


[Spatial split] axis=lon  train=917532  val=114692  test=114691


C:\Users\Ahmed sameh\AppData\Local\Temp\ipykernel_1464\4183623045.py:53: RuntimeWarning: Mean of empty slice
  self.mu = np.nanmean(x)
c:\Users\Ahmed sameh\AppData\Local\Programs\Python\Python313\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2015: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


duplicate directed edges in line-graph edge_index: 0
Train graph: 917532 nodes | 3971562 edges
Val graph:   114692 nodes | 505405 edges
Test graph:  114691 nodes | 481319 edges
Degree stats:
  Train: {'num_nodes': 917532, 'isolated': 1497, 'deg1': 6942, 'mean_deg': 4.328526973724365}
  Val:   {'num_nodes': 114692, 'isolated': 111, 'deg1': 726, 'mean_deg': 4.406628131866455}
  Test:  {'num_nodes': 114691, 'isolated': 132, 'deg1': 1086, 'mean_deg': 4.196659088134766}


C:\Users\Ahmed sameh\AppData\Local\Temp\ipykernel_1464\4183623045.py:567: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  out["wid_mae_m"] = float(torch.mean(torch.abs(pred["width"][masks["wid"]] - data.y_width[masks["wid"]]))) \


Epoch 0001 | total=37.9938 | hwy=1.770, lan=1.087, onw=0.649, wid=9.005 max=25.483 min=0.000 | VAL fixed: hwy_F1=0.191, lan_F1=0.298, onw_AUROC=0.474, wid_MAE(m)=12.439 max_MAE=26.701, min_MAE=nan | log_vars=[ 0.001  0.001 -0.001  0.001  0.001 -0.001]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.17 GiB. GPU 0 has a total capacity of 6.00 GiB of which 0 bytes is free. Of the allocated memory 10.77 GiB is allocated by PyTorch, and 1.76 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# =========================
# CELL X) Save trained model + preprocessing artifacts (for another city)
# =========================
import os
import json
import torch

SAVE_DIR = "./checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)

ckpt_path = os.path.join(SAVE_DIR, "nyc_gat_multitask.pt")

checkpoint = {
    # model
    "model_state": model.state_dict(),

    # model config needed to recreate the architecture
    "model_cfg": {
        "num_highway": int(num_highway),
        "hwy_emb_dim": 16,
        "lanes_emb_dim": 8,
        "oneway_emb_dim": 4,
        "cont_dim": 12,
        "hidden": 32,
        "heads": 2,
        "dropout": 0.1,
    },

    # categorical vocab mapping (CRITICAL for cross-city)
    "hwy2id": hwy2id,
    "id2hwy": id2hwy,

    # special token IDs
    "token_ids": {
        "HIGHWAY_MASK_ID": int(HIGHWAY_MASK_ID),
        "LANES_MASK_ID": int(LANES_MASK_ID),
        "LANES_MISS_ID": int(LANES_MISS_ID),
        "ONEWAY_MASK_ID": int(ONEWAY_MASK_ID),
        "ONEWAY_MISS_ID": int(ONEWAY_MISS_ID),
    },

    # scalers (CRITICAL for consistent normalization)
    # All four are fit on the Jakarta training split and MUST be saved —
    # the model expects z-scored length / width / max_speed / min_speed on input.
    "scalers": {
        "len_mu": float(len_scaler.mu),
        "len_sd": float(len_scaler.sd),
        "wid_mu": float(wid_scaler.mu),
        "wid_sd": float(wid_scaler.sd),
        "max_mu": float(max_scaler.mu),
        "max_sd": float(max_scaler.sd),
        "min_mu": float(min_scaler.mu),
        "min_sd": float(min_scaler.sd),
    },

    # metadata (optional)
    "meta": {
        "seed": int(SEED),
        "split_axis": "lon",
        "p_mask": float(P_MASK),
    }
}

torch.save(checkpoint, ckpt_path)
print(f"Saved checkpoint to: {ckpt_path}")

Saved checkpoint to: ./checkpoints\jakarta_gat_multitask.pt
